# 🍔 Food-101 Fine-Tuning (TFDS + EfficientNet)

This notebook:
- Loads your **pre-trained EfficientNet model** from Kaggle Models
- Uses **TFDS Food-101** (clean, no filesystem issues)
- Fine-tunes top layers safely
- Saves final model to `/kaggle/working`


In [ ]:
import os
import tensorflow as tf
import tensorflow_datasets as tfds

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 📦 Load Model from Kaggle Input

In [ ]:
MODEL_INPUT_PATH = '/kaggle/input/best-model/best_model.keras'

assert os.path.exists(MODEL_INPUT_PATH), '❌ Model file not found'
print('✅ Found model at:', MODEL_INPUT_PATH)

model = tf.keras.models.load_model(MODEL_INPUT_PATH)
model.summary()

## 🍱 Load Food-101 via TFDS

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

(train_ds, val_ds), info = tfds.load(
    'food101',
    split=['train', 'validation'],
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = info.features['label'].num_classes
print('Classes:', NUM_CLASSES)


## 🧼 Preprocessing Pipeline

In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = (
    train_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(2048)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


## 🔓 Fine-Tune Top Layers Only (Safe Mode)

In [ ]:
FINE_TUNE_AT = int(len(model.layers) * 0.75)

for layer in model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

for layer in model.layers[FINE_TUNE_AT:]:
    layer.trainable = True

print(f'🔓 Fine-tuning from layer {FINE_TUNE_AT} / {len(model.layers)}')


## ⚙️ Compile (Low LR for Fine-Tuning)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        'accuracy',
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)
    ]
)


## 🧠 Callbacks

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/kaggle/working/best_model.keras',
        save_best_only=True
    )
]


## 🚀 Fine-Tune

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)


## 💾 Save Final Model

In [ ]:
FINAL_PATH = '/kaggle/working/food101_efficientnet_finetuned.keras'
model.save(FINAL_PATH)

print('✅ Final model saved:', FINAL_PATH)
print('📦 Output files:', os.listdir('/kaggle/working'))
